# How to use this cookbook

This is the **living handbook for engram** - a reference I keep coming back to and extend as I learn. It covers two things:

- **How the repo works** - structure, conventions, and the render/publish workflow.
- **How to author a notebook** - the Quarto features worth using.

For every visual feature in Part 2 you get two blocks: a **Source** block (the markup that produces it) and a **Renders as** block (what Quarto produces from it). Render locally with `quarto preview` to see the rendered halves live.

When you adopt something new, add a section and append a line to the [Update log](#update-log).

::: {.callout-note}
A `.ipynb` carries Quarto front matter in a **raw cell** at the very top (the cell above this one). In a `.qmd` file the same YAML sits between `---` fences.
:::


# Part 1 - The repo

## Structure

One folder per project; each holds its notebooks. Site config and the landing page live at the root.

```
engram/
├── _quarto.yml              # site config (theme, navbar, frozen builds)
├── index.qmd                # landing page + auto-listing of every notebook
├── references.bib           # shared bibliography for @citations
├── cookbook.ipynb           # this handbook
├── braindecode/
│   └── shallow-convnet.ipynb
└── _freeze/                 # committed precomputed outputs (do not delete)
```

Build artifacts - `_site/` and `.quarto/` - are git-ignored and regenerate on every render. `_freeze/` is the exception: it **is** committed, because the published site is assembled from it.


## Adding a new notebook

1. Drop the `.ipynb` into the relevant project folder (or create a new folder).
2. Add a front-matter raw cell at the top (see the [starter](#copy-paste-starter)).
3. Add a Colab badge as the first markdown cell.
4. Cite any papers with `@key` and add the entries to `references.bib`.
5. `quarto render` locally so `_freeze/` is refreshed, then commit and push.

The notebook then appears automatically on the homepage listing - no manual index edit.


## Render & publish workflow

Builds are **frozen**: GitHub Actions never executes notebooks, it only assembles the site from the committed `_freeze/` cache. That keeps CI fast and dependency-free - but it means *you* must render locally before pushing.

```bash
quarto preview        # live preview while editing
quarto render         # refreshes _freeze/ with fresh outputs
git add . && git commit -m "Add notebook" && git push
```

Pushing to `main` triggers the Action, which publishes to the `gh-pages` branch -> GitHub Pages at `bkowshik.github.io/engram`.


# Part 2 - Authoring with Quarto

Each section below pairs the **Source** (what you write) with **Renders as** (what Quarto produces).


## Front matter & metadata

Every notebook opens with a raw cell of YAML. This drives the title, the listing entry, categories, and per-document options. (The "rendered" effect here is the page's title block and listing entry, not inline output.)

**Source**

````yaml
---
title: "Short, specific title"
description: "One line - shown in the listing and the social card."
date: "2026-06-16"
date-modified: last-modified   # auto-updates from file mtime - good for living docs
categories: [eeg, pytorch, bci] # power the tag filter + listing
bibliography: references.bib     # enables @citations
toc: true
code-tools: true                # adds the "view source / copy" menu
image: thumbnail.png            # optional - used in the social preview card
---
````


## Callouts

Coloured boxes for asides, insights, and gotchas. Flavours: `note`, `tip`, `warning`, `important`, `caution`.

**Source**

````md
::: {.callout-tip}
## Key insight
The takeaway in one or two sentences.
:::

::: {.callout-warning collapse="true"}
## Gotcha (collapsed by default)
Add `collapse="true"` to fold long asides.
:::
````

**Renders as**

::: {.callout-tip}
## Key insight
The takeaway in one or two sentences.
:::

::: {.callout-warning collapse="true"}
## Gotcha (collapsed by default)
Add `collapse="true"` to fold long asides.
:::


## Math

**Source**

````md
Inline: $h_t = \sigma(W x_t + b)$

$$
\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} -\,y_i \log \hat{y}_i
$$ {#eq-nll}

Reference it with `@eq-nll`.
````

**Renders as**

Inline: $h_t = \sigma(W x_t + b)$

$$
\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} -\,y_i \log \hat{y}_i
$$ {#eq-nll}

The negative log-likelihood in @eq-nll is the loss the Shallow ConvNet trains with.


## Citations & bibliography

With `bibliography: references.bib` in the front matter, cite inline and Quarto builds the reference list automatically.

**Source**

````md
The Shallow ConvNet comes from @schirrmeister2017.
A parenthetical citation looks like [@schirrmeister2017].
````

...resolving against this `.bib` entry:

````bibtex
@article{schirrmeister2017,
  title   = {Deep learning with convolutional neural networks for EEG decoding and visualization},
  author  = {Schirrmeister, Robin Tibor and others},
  journal = {Human Brain Mapping}, year = {2017}, volume = {38}, pages = {5391--5420}
}
````

**Renders as**

The Shallow ConvNet comes from @schirrmeister2017. A parenthetical citation looks like [@schirrmeister2017]. (The full reference appears in the auto-generated list at the end of the page.)


## Diagrams (Mermaid & Graphviz)

Diagram-as-code - perfect for an architecture sketch instead of an ASCII block.

**Source** (4 outer backticks so the inner block shows literally)

`````md
```{mermaid}
flowchart LR
  X["EEG (22 ch x 1125)"] --> T["Temporal conv"]
  T --> S["Spatial conv"] --> P["Square -> pool -> log"] --> C["Classifier"]
```
`````

**Renders as**

```{mermaid}
flowchart LR
  X["EEG (22 ch x 1125)"] --> T["Temporal conv"]
  T --> S["Spatial conv"] --> P["Square -> pool -> log"] --> C["Classifier"]
```


## Code cells: source + output together

A code cell is the clearest case: by default Quarto shows **both** the source and the output it produces. The cell below is labelled and captioned so the figure can be cross-referenced - and its code stays visible above the plot.

In [ ]:
#| label: fig-signal
#| fig-cap: "A 10 Hz sine wave - a stand-in for a band-limited neural signal."
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 1, 500)
x = np.sin(2 * np.pi * 10 * t)

plt.figure(figsize=(6, 2.4))
plt.plot(t, x)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()


@fig-signal shows the rendered output above, directly under the code that produced it. Control what each cell reveals with cell options:

| Option | Effect |
|---|---|
| `#| echo: false` | hide the code, keep the output |
| `#| output: false` | run the code, hide the output |
| `#| code-fold: true` | keep the code but collapse it behind a toggle |
| `#| label: fig-x` / `tbl-x` | make it cross-referenceable (the `fig-`/`tbl-` prefix matters) |
| `#| fig-cap: "..."` | figure caption |
| `#| warning: false` | suppress warnings in the render |

The default (`echo: true`) is exactly the "show both" behaviour - use the options above only when you deliberately want to hide one half.


## Cross-references

**Source**

````md
See @fig-signal and @eq-nll. Sections work too: tag a heading `{#sec-setup}`, then `@sec-setup`.
````

**Renders as**

See @fig-signal and @eq-nll. Quarto numbers and links them automatically, so references never go stale when you reorder.


## Tabsets

Stack alternatives behind tabs - handy for "same idea, two frameworks".

**Source**

````md
::: {.panel-tabset}
## NumPy
```python
import numpy as np
x = np.zeros((22, 1125))
```
## PyTorch
```python
import torch
x = torch.zeros(22, 1125)
```
:::
````

**Renders as**

::: {.panel-tabset}

## NumPy

```python
import numpy as np
x = np.zeros((22, 1125))
```

## PyTorch

```python
import torch
x = torch.zeros(22, 1125)
```

:::


## Figures, captions & lightbox

**Source**

````md
![Shallow ConvNet, Figure 2 from the paper.](architecture.png){#fig-arch}
````

Enable click-to-zoom for all images via `_quarto.yml`:

````yaml
format:
  html:
    lightbox: true
````

**Renders as:** a captioned, numbered figure (`Figure 1: ...`) that is cross-referenceable with `@fig-arch` and opens in a zoom overlay on click.


## Listings, categories & search (site-level)

These live in `_quarto.yml` / `index.qmd`, but every notebook feeds them:

- **Listing** - `index.qmd` auto-indexes every `**/*.ipynb`, sorted by `date`.
- **Categories** - the `categories:` in each notebook become clickable filters.
- **Full-text search** - on by default; searches every notebook, client-side, free on GitHub Pages.


## Social cards, comments & formats (site-level config)

These are `_quarto.yml` settings rather than inline output:

````yaml
website:
  open-graph: true        # rich link previews when shared
  twitter-card: true
  comments:
    giscus:               # discussion thread per post, backed by GitHub Discussions
      repo: bkowshik/engram

format:
  html: default
  revealjs:               # a speaker deck from the same notebook cells
    slide-level: 2
````

Render a slide deck from any notebook with `quarto render notebook.ipynb --to revealjs`.


## Freeze & cache (reproducibility)

engram uses **frozen** builds, set in `_quarto.yml`:

````yaml
execute:
  freeze: auto   # never re-execute on CI; rebuild only when the source changes
````

You render locally -> `_freeze/` is refreshed -> commit the notebook **and** `_freeze/` -> the Action assembles the site without executing (no data, no GPU, no deps). Add `execute: cache: true` for local incremental caching of expensive cells.


## Colab badge

**Source** (first markdown cell of a notebook)

````md
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bkowshik/engram/blob/main/FOLDER/NOTEBOOK.ipynb)
````

**Renders as:** a clickable "Open In Colab" badge that launches the notebook in Google Colab.


# Copy-paste starter {#copy-paste-starter}

Front matter for a new notebook (raw cell at the top):

````yaml
---
title: ""
description: ""
date: "YYYY-MM-DD"
date-modified: last-modified
categories: []
bibliography: references.bib
toc: true
code-tools: true
---
````

**Before you push:**

- [ ] Front matter filled in (title, description, date, categories)
- [ ] Colab badge points at the right folder/filename
- [ ] Papers cited with `@key`, entries added to `references.bib`
- [ ] Cells reveal what you intend (`echo`/`output`/`code-fold`)
- [ ] `quarto render` run locally so `_freeze/` is fresh
- [ ] No private references (paths, personal notes)


# Update log {#update-log}

Append a line whenever you learn or adopt something new.

| Date | Change |
|---|---|
| 2026-06-16 | Created. Each Part 2 feature now shows Source + Renders-as; example plot cell shows code and output together. |
